# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists('FlyRank_AI'):
    !git clone -q https://github.com/AhmedMahmoud-123/FlyRank_AI.git
os.chdir('FlyRank_AI')
!python scripts/01_prepare_features.py

import pandas as pd
import numpy as np
df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows')

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/data/processed/refresh_feature_vector.csv
30,000 rows


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
traffic_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']
print(df[traffic_cols].describe(percentiles=[.5, .9, .99]))

# heavy-tail check: mean >> median is the signature
for col in traffic_cols:
    print(f"{col:20} mean={df[col].mean():>10.1f}   median={df[col].median():>8.1f}   max={df[col].max():>10.0f}")

       impressions_90d    clicks_90d  sessions_90d  ai_sessions_90d
count     30000.000000  30000.000000  30000.000000     30000.000000
mean       5200.366300     16.097333     37.066633         0.204500
std       16838.019547     75.076958    107.069131         1.363601
min           1.000000      0.000000      1.000000         0.000000
50%         731.000000      1.000000      7.000000         0.000000
90%       12136.400000     32.000000     88.000000         0.000000
99%       73505.830000    253.010000    451.010000         5.000000
max      517715.000000   4178.000000   4345.000000        64.000000
impressions_90d      mean=    5200.4   median=   731.0   max=    517715
clicks_90d           mean=      16.1   median=     1.0   max=      4178
sessions_90d         mean=      37.1   median=     7.0   max=      4345
ai_sessions_90d      mean=       0.2   median=     0.0   max=        64


**Distributions:** [fill after running — traffic columns are almost certainly heavy-tailed
(mean far above median, huge max). This means raw Pearson correlation later would be dominated
by a handful of giant pages — log-transform or use grouped medians instead, per the skill.]

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [4]:
def verdict_table(df, group_col, metric_col, min_n=50):
    t = df.groupby(group_col)[metric_col].agg(['median', 'count'])
    t['flag'] = np.where(t['count'] < min_n, 'INSUFFICIENT DATA', '')
    return t.sort_values('median', ascending=False)

# Signal test 1: does word count relate to impressions? (mirrors the paper's Myth 3)
print(verdict_table(df, 'word_count_tier', 'impressions_90d'))

# Signal test 2: does freshness relate to trend? (mirrors the paper's Finding 4)
print(pd.crosstab(df['freshness_tier'], df['trend_direction'], normalize='index').round(3))

# Signal test 3: does competition level relate to declining rate? (mirrors Myth 5)
print(df.groupby('competition_level')['is_declining_label'].agg(['mean', 'count']))

                 median  count flag
word_count_tier                    
3500+            1340.0   6285     
2000-3500         997.0  11263     
unknown           878.0   7699     
1000-2000         172.0   3780     
<1000               4.0    973     
trend_direction   down   flat    new  stable     up
freshness_tier                                     
0-30             0.511  0.044  0.104   0.186  0.155
181+             0.471  0.092  0.144   0.138  0.155
31-90            0.589  0.006  0.040   0.149  0.217
91-180           0.611  0.026  0.008   0.229  0.126
                       mean  count
competition_level                 
HIGH               0.556057   2658
LOW                0.563199  22896
MEDIUM             0.566993   1836
unknown            0.324904   2610


**Signal test 1 — "Longer content gets more impressions."**
Test: median `impressions_90d` by `word_count_tier`, with counts.
Verdict: [fill in — CONFIRMED / OPPOSITE / MIXED / FALSE, one sentence why, and flag any
bucket under 50 rows as insufficient rather than a real verdict]

**Signal test 2 — "Fresher content trends up more."**
Test: `trend_direction` distribution within each `freshness_tier`.
Verdict: [fill in]

**Signal test 3 — "High-competition keywords decline more often."**
Test: `is_declining_label` rate by `competition_level`, n shown.
Verdict: [fill in]

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [5]:
striking = df[df['position_tier'].str.contains('striking', case=False, na=False)]
print(f"striking-distance pages: {len(striking):,}")
print(striking['is_declining_label'].mean(), "declining rate")
print(df['is_declining_label'].mean(), "overall declining rate")

striking-distance pages: 7,304
0.6095290251916758 declining rate
0.5420666666666667 overall declining rate


**Flag-linked test:** FlyRank's product flags treat striking-distance pages (position 11-20) as
high-priority recovery targets — the assumption being these pages are close to a breakthrough,
not in freefall. Test: declining rate for striking-distance pages vs. the overall rate.
Verdict: [fill in — does the data support treating "close to page 1" as "worth prioritizing,"
or do striking-distance pages actually decline at the same or higher rate than everything else?]

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

[Two or three sentences, written after seeing the real verdicts — e.g. "Word count alone
doesn't predict impressions in this dataset, so a content team shouldn't pad pages to hit a
length target. Freshness does correlate with upward trend, supporting the refresh-first
priority from Week 4's baseline rule."]

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.